In [ ]:
# Groq 무료모델을 이용한다.
# OpenAI나 Claude같은 상용 모델은 유료계정 api_key가 필요하다
#

In [32]:
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import psutil
import os
from openai import OpenAI
from IPython.display import Image
from typing import TypedDict, Annotated
from operator import add
from dotenv import load_dotenv

In [33]:
# 환경변수

load_dotenv()



True

In [34]:
# groq 잘 작동하는지 테스트

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": "안녕, 한 줄로 답해"
            }
        ]
    )

    print("연결 성공")
    print(response.choices[0].message.content)

except Exception as e:
    print("에러 발생")
    print(type(e).__name__)
    print(e)

연결 성공
안녕하세요


In [35]:


MAX_STEP = 5

# ===== LLM =====

llm = ChatOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model="llama-3.3-70b-versatile",
    temperature=0
)


# ===== Tool =====

@tool
def get_cpu_usage():
    """현재 CPU 사용률 조회"""
    value = psutil.cpu_percent()

    print(f"[TOOL] CPU 조회: {value}")

    return f"CPU: {value}%"


@tool
def get_memory_usage():
    """현재 메모리 사용률"""

    value = psutil.virtual_memory().percent

    print(f"[TOOL] MEM 조회: {value}")

    return f"MEMORY: {value}%"

@tool
def free_memory():
    """
    메모리 사용량이 높을 때 실행.
    메모리 40% 이상이면 사용 고려.
    """

    print(
        "[TOOL] 메모리 정리"
    )

    import gc

    gc.collect()

    return "메모리 정리 완료"

tools = [
    get_cpu_usage,
    get_memory_usage,
    free_memory
]


llm_with_tools = llm.bind_tools(
    tools
)


# ===== 상태 =====

class AgentState(TypedDict):
    # messages: Annotated[list, add]
    messages: list
    step_count: int


# ===== 노드 =====

def agent_node(state):

    print("\n===== AGENT =====")

    step = state.get(
        "step_count",
        0
    )

    print(f"입력:{state['messages'][-1]}")
    print(f"스텝:{step}")

    result = llm_with_tools.invoke(
        state["messages"]
    )

    print("\nLLM 응답:")

    print(result)

    return {
        "messages":[result],
        "step_count":step+1
    }


tool_node = ToolNode(tools)


# tool 호출 여부 판단
def should_continue(state):

    step = state.get("step_count",0)

    print(f"STEP={step}")

    if step >= MAX_STEP:
        print("10회 초과 종료")
        return END

    msg = state["messages"][-1]

    if (hasattr(msg,"tool_calls") and msg.tool_calls):
        return "tools"

    return END


# ===== Graph 생성 =====

graph = StateGraph(
    AgentState
)

graph.add_node(
    "agent",
    agent_node
)

graph.add_node(
    "tools",
    tool_node
)

graph.set_entry_point(
    "agent"
)

graph.add_conditional_edges(
    "agent",
    should_continue
)

graph.add_edge(
    "tools",
    "agent"
)


app = graph.compile()

# ===== Graph 구조 출력 =====

print("\n===== GRAPH =====")

# print(
#     app.get_graph()
# )

print(
    app.get_graph()
       .draw_mermaid()
)

# Image(
#     app.get_graph()
#     .draw_mermaid_png()
# )


# ===== 실행 =====

inputs={

    "messages":[
        ("user",
         """
         CPU와 메모리 상태 확인하고
         문제가 있으면 분석해줘
          """)
    ],
    "step_count":0
}


print(
    "\n===== START ====="
)

for event in app.stream(inputs):

    print("\nEVENT:")

    print(event)



print(
    "\n===== END ====="
)


===== GRAPH =====
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	agent(agent)
	tools(tools)
	__end__(<p>__end__</p>)
	__start__ --> agent;
	agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


===== START =====

===== AGENT =====
입력:('user', '\n         CPU와 메모리 상태 확인하고\n         문제가 있으면 분석해줘\n          ')
스텝:0

LLM 응답:
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 347, 'total_tokens': 366, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'queue_time': 0.072106666, 'prompt_time': 0.032274687, 'completion_time': 0.047522597, 'total_time': 0.079797284}, 'model_provider': 'openai', 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'id': 'chatcmpl-30c34265-4b6e-4371-90f7-9408c82ac2a9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs